# GeoNUSAF — Stage-3 layout-fidelity RESCORE (inference only, no training)

**What this notebook does, and why it exists.**

Every contamination number in the Results/Discussion master (§22.7, Table 11.5d) traces back to a
single measurement: the per-class **layout IoU** of the synthetic pool `fake_pairs_R1`, reported as
**River 0.033–0.055**. That was measured with the part-1 fold-1 **SegFormer-B0** scorer — an
instrument that reaches only **0.1434 River IoU on real fold-1 imagery**. So the tool that declared
synthetic River unrecoverable can barely recover *real* River. The reading sits near that
instrument's floor, and a floor-bound measurement cannot support the proportionality argument the
discussion leans on.

This notebook re-runs **exactly the same evaluation** with scorers that can actually see River:

| Scorer | River IoU on real fold-1 | Role |
|---|---|---|
| SegFormer-B0 (part-1 fold-1) | 0.1434 | **control** — must reproduce the published numbers |
| SegNeXt-T (Stage-4 arm R0) | 0.4594 | **primary** new instrument (3.2× the control) |
| TC-SegFormer (Stage-4 arm R0) | 0.4761 | **optional** — needs the model class pasted in §13 |

It also pulls every `results_*.json` off the Hub (§23), which is free and supplies the four metric
families missing from every model card — including **clDice for Road and River**, which bears
directly on the soft-clDice hypothesis in §22.10.

**No training. No gradients. No new data.** Two forward passes per image over 804 synthetic pairs
and 136 real validation tiles, per scorer. Expect **10–20 minutes** on a T4 for all three scorers.

---

### What to run and what to send back

1. Run every cell top to bottom. Kaggle: **GPU T4 ×1**, **Internet ON**.
2. If §14 reports that a scorer failed to build, that is fine — the notebook degrades and continues
   on whichever scorers did build. SegNeXt alone is enough to answer the question.
3. Send back **`rescore_layout_fidelity.json`** and **`rescore_summary.csv`** from §24, plus the
   printed table in §21. That is everything needed to update the master document.

### The one thing to check before trusting the output

§21 compares the control scorer against the published Stage-3 numbers. **If the control does not
reproduce mIoU ≈ 0.4395 on the fakes and ≈ 0.4646 on real val, the harness is wrong and the new
numbers mean nothing.** That check is the gate; do not read past it if it fails.

### Honest limitation, stated up front

SegNeXt-T R0 and TC-SegFormer R0 were trained on the same 268 fold-1 tiles that trained the
generators. Generator–scorer collusion therefore applies to them exactly as it did to B0. Swapping
the scorer buys **sensitivity**, not independence. This is recorded in the master doc and must stay
in the paper.

## 1. Install

Pinned the same way the Stage-4 notebooks pin it: `transformers<5`, because 5.x renamed
`decode_head.linear_c` to `linear_projections` and that changes every checkpoint key.

In [ ]:
!pip -q install "transformers<5" "huggingface_hub>=0.23" timm scikit-image kagglehub 2>&1 | tail -3
print("install done")

## 2. Imports and environment

In [ ]:
import os, sys, json, glob, math, time, hashlib, shutil, tarfile, traceback, warnings
from dataclasses import dataclass, field, asdict
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
warnings.filterwarnings("ignore")
Image.MAX_IMAGE_PIXELS = None

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
try:
    import transformers; print("transformers", transformers.__version__)
except Exception as e:
    print("transformers import failed:", e)

## 3. CFG — single source of truth

Same `@dataclass CFG` pattern as every other GeoNUSAF notebook. Nothing below this cell should hold
a hard-coded path or hyper-parameter.

In [ ]:
@dataclass
class CFG:
    # ---- data sources -------------------------------------------------------
    KAGGLE_REAL: str   = "sugam24/satellite-datasets"
    KAGGLE_FAKE: str   = "sugam24/geonusaf-fake-pairs-block-fold1"
    HF_FAKE_REPO: str  = "sugam24/geonusaf-stage3-fakepairs-block-fold1"   # fallback
    FAKE_SET: str      = "fake_pairs_R1"      # the ONLY set Stage 4 consumed
    # ---- split --------------------------------------------------------------
    N_BASE: int        = 408
    N_FOLDS: int       = 3
    FOLD: int          = 1
    BUFFER: int        = 2                     # dropped from TRAIN only
    EXPECT_TRAIN: int  = 268
    EXPECT_VAL: int    = 136
    TRAIN_SHA1_REF: str = "eb0aebdd991971c319fc5fde92f8ad4aa3d82b65"
    # ---- eval ---------------------------------------------------------------
    IMG_SIZE: int      = 512
    NUM_CLASSES: int   = 6
    IGNORE_INDEX: int  = 255
    BATCH: int         = 8
    AMP: bool          = True
    NUM_WORKERS: int   = 2
    CLDICE_CLASSES: tuple = (1, 2)             # Road, River
    # ---- scorers ------------------------------------------------------------
    RUN_B0_CONTROL: bool   = True              # gate: must reproduce Stage-3
    RUN_SEGNEXT: bool      = True              # primary new instrument
    RUN_TC: bool           = True              # needs §13 paste-in; skipped if absent
    REPO_B0: str      = "Pranilllllll/segformer-b0-kaggle-ktm-dataset-block-fold1"
    REPO_SEGNEXT: str = "sugam24/geonusaf-s4-segnext-t-R0-block-fold1"
    REPO_TC: str      = "sugam24/geonusaf-s4-tcsegformer-R0-block-fold1"
    # ---- published Stage-3 reference values (the gate in §21) ---------------
    REF_FAKE_MIOU: float  = 0.4395
    REF_REAL_MIOU: float  = 0.4646
    REF_TOL: float        = 0.02
    # ---- Stage-4 repos to harvest results_*.json from (§23) -----------------
    S4_REPOS: tuple = (
        "Pranilllllll/geonusaf-s4-unet-r18-R0-block-fold1",
        "Pranilllllll/geonusaf-s4-unet-r18-R1-block-fold1",
        "Pranilllllll/geonusaf-s4-unetformer-r18-R0-block-fold1",
        "Pranilllllll/geonusaf-s4-unetformer-r18-R1-block-fold1",
        "Pranilllllll/geonusaf-s4-segformer-b0-R0-block-fold1",
        "Pranilllllll/geonusaf-s4-segformer-b0-R1-block-fold1",
        "sugam24/geonusaf-s4-segnext-t-R0-block-fold1",
        "sugam24/geonusaf-s4-segnext-t-R1-block-fold1",
        "sugam24/geonusaf-s4-tcsegformer-R0-block-fold1",
        "sugam24/geonusaf-s4-tcsegformer-R1-block-fold1",
    )
    # ---- output -------------------------------------------------------------
    OUT_DIR: str      = "/kaggle/working/rescore"
    HF_TOKEN_SECRET: str = "HF_TOKEN"          # optional Kaggle secret
    SEED: int         = 42

cfg = CFG()
Path(cfg.OUT_DIR).mkdir(parents=True, exist_ok=True)
CLASS_NAMES = ["Residential", "Road", "River", "Forest", "UnusedLand", "Agricultural"]
print(json.dumps({k: v for k, v in asdict(cfg).items() if not isinstance(v, tuple)}, indent=2)[:1200])

## 4. Device, seeding, determinism OFF

`torch.use_deterministic_algorithms(True)` costs ~6× on this workload (measured, project-wide rule).
This is inference, so it buys nothing at all here.

In [ ]:
np.random.seed(cfg.SEED); torch.manual_seed(cfg.SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEV)
if DEV.type == "cuda":
    print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret(cfg.HF_TOKEN_SECRET)
    print("HF token loaded from Kaggle secrets")
except Exception:
    print("no HF token (fine for public repos)")

## 5. Acquire the data

Real tiles from the Kaggle dataset; synthetic pairs from the Kaggle mirror, falling back to the Hub
tarball chunks if the mirror is not attached. Both paths end at the same place: a directory holding
`fake_pairs_R1/` with paired image/mask files.

In [ ]:
import kagglehub
from huggingface_hub import hf_hub_download, list_repo_files, snapshot_download

def find_dir(root, name):
    for p in Path(root).rglob(name):
        if p.is_dir(): return p
    return None

# ---- real -----------------------------------------------------------------
REAL_ROOT = None
for cand in ["/kaggle/input/satellite-datasets", "/kaggle/input"]:
    d = find_dir(cand, "images")
    if d is not None and find_dir(d.parent, "masks") is not None:
        REAL_ROOT = d.parent; break
if REAL_ROOT is None:
    print("downloading real dataset via kagglehub ...")
    REAL_ROOT = Path(kagglehub.dataset_download(cfg.KAGGLE_REAL))
    REAL_ROOT = find_dir(REAL_ROOT, "images").parent
REAL_IMG = REAL_ROOT / "images"; REAL_MSK = REAL_ROOT / "masks"
print("REAL_ROOT :", REAL_ROOT, "|", len(list(REAL_IMG.glob('*.png'))), "images")

# ---- fake -----------------------------------------------------------------
FAKE_DIR = None
for cand in ["/kaggle/input", "/kaggle/working"]:
    d = find_dir(cand, cfg.FAKE_SET)
    if d is not None: FAKE_DIR = d; break
if FAKE_DIR is None:
    try:
        print("downloading fake pairs via kagglehub ...")
        p = Path(kagglehub.dataset_download(cfg.KAGGLE_FAKE))
        FAKE_DIR = find_dir(p, cfg.FAKE_SET)
    except Exception as e:
        print("kagglehub fake mirror failed:", e)
if FAKE_DIR is None:
    print("falling back to HF tarball chunks ...")
    dst = Path("/kaggle/working/_fake"); dst.mkdir(parents=True, exist_ok=True)
    files = list_repo_files(cfg.HF_FAKE_REPO, repo_type="dataset", token=HF_TOKEN)
    chunks = [f for f in files if f.endswith(".tar.gz") and "_R1_" in f]
    if not chunks:
        chunks = [f for f in files if f.endswith(".tar.gz")]
    for f in chunks:
        lp = hf_hub_download(cfg.HF_FAKE_REPO, f, repo_type="dataset", token=HF_TOKEN)
        with tarfile.open(lp) as t: t.extractall(dst)
    FAKE_DIR = find_dir(dst, cfg.FAKE_SET)

assert FAKE_DIR is not None, "could not locate fake_pairs_R1 — attach the Kaggle dataset or check the HF repo"
print("FAKE_DIR  :", FAKE_DIR, "|", len(list(FAKE_DIR.rglob('*.png'))), "png files")

## 6. Palette LUT — disambiguated by class share, not by unmapped pixels

The project palette is recorded in **BGR** order, and it has a nasty property: it is **closed under
channel reversal**. `(128,0,0)` reversed is `(0,0,128)` — Residential becomes River — and
`(0,128,128)` reversed is `(128,128,0)` — Forest becomes UnusedLand. `(0,128,0)` and `(128,0,128)`
are palindromes. So *both* orderings map 100 % of pixels and an unmapped-pixel test cannot tell them
apart, while getting it wrong silently swaps the two classes this entire notebook is about.

The disambiguation therefore uses the **known class marginal** instead: Residential is 52–56 % of
pixels and River is ~0.5 %. Whichever ordering puts Residential first and River last is the right
one. That test cannot be fooled by the palindrome structure.

In [ ]:
PAL_BGR = {(128,0,0):0,(0,128,0):1,(0,0,128):2,(0,128,128):3,(128,128,0):4,(128,0,128):5,(0,0,0):255}

def build_lut(pal, swap):
    lut = np.full((256,256,256), 255, np.uint8)
    for (a,b,c), v in pal.items():
        r,g,bl = (c,b,a) if swap else (a,b,c)
        lut[r,g,bl] = v
    return lut

LUT_A, LUT_B = build_lut(PAL_BGR, False), build_lut(PAL_BGR, True)

def _unmapped(lut, paths):
    bad = tot = 0
    for p in paths:
        m = np.array(Image.open(p).convert("RGB").resize((256,256), Image.NEAREST))
        idx = lut[m[...,0], m[...,1], m[...,2]]
        bad += int((idx == 255).sum()); tot += idx.size
    return bad / max(tot,1)

RNG_LUT = np.random.default_rng(cfg.SEED)

def _share(lut, paths):
    cnt = np.zeros(6, np.int64)
    for p in paths:
        m = np.array(Image.open(p).convert("RGB").resize((256,256), Image.NEAREST))
        idx = lut[m[...,0], m[...,1], m[...,2]]
        cnt += np.bincount(idx[idx != 255].ravel(), minlength=6)
    return cnt / max(cnt.sum(), 1)

# Sample RANDOM base tiles, not a lexicographic window. sorted(glob)[:24] returns
# mask1, mask1hf, mask1vf, mask10, mask100, ... — about 8 base tiles, whose marginal
# is nothing like the global one. 120 base tiles is representative enough to decide.
all_ids = [i for i in range(1, cfg.N_BASE+1) if (REAL_MSK/f"mask{i}.png").exists()]
pick    = RNG_LUT.choice(len(all_ids), size=min(120, len(all_ids)), replace=False)
sample  = [REAL_MSK/f"mask{all_ids[j]}.png" for j in pick]
print(f"palette check over {len(sample)} randomly chosen base tiles (not a lexicographic window)\n")

ua, ub = _unmapped(LUT_A, sample), _unmapped(LUT_B, sample)
sa, sb = _share(LUT_A, sample),    _share(LUT_B, sample)
print(f"unmapped share   as-written {ua:.5f}   swapped {ub:.5f}")
print("  ^ identical by construction. The palette is closed under channel reversal:")
print("    (128,0,0)<->(0,0,128) swaps Residential/River and (0,128,128)<->(128,128,0)")
print("    swaps Forest/UnusedLand, while (0,128,0) and (128,0,128) are palindromes.")
print("    So this test can never discriminate. The class marginal below is what decides.\n")

# Exhaustive global shares over all 408 tiles @512 (the project's measured reference).
EXPECT = [56.02, 1.57, 0.60, 10.38, 12.97, 18.10]
print(f"{'class':14s} {'as-written%':>12s} {'swapped%':>10s} {'expected%':>10s}")
for i, c in enumerate(CLASS_NAMES):
    print(f"{c:14s} {100*sa[i]:12.3f} {100*sb[i]:10.3f} {EXPECT[i]:10.3f}")

e      = np.array(EXPECT)/100.0
da, db = float(np.abs(sa-e).sum()), float(np.abs(sb-e).sum())
SWAP   = db < da
LUT    = LUT_B if SWAP else LUT_A
chosen = sb if SWAP else sa
print(f"\nL1 to the global marginal:  as-written {da:.4f}   swapped {db:.4f}")
print(f"margin |da-db| = {abs(da-db):.4f}   ->  SWAP = {SWAP}")

# Gate. Thresholds are deliberately loose: they must pass on any representative
# sample and still reject the reversed ordering, which puts ~56% into River.
ok_res  = chosen[0] > 0.35     # Residential is 56% globally; never below 35% at n=120
ok_riv  = chosen[2] < 0.05     # River is 0.6% globally; never above 5% at n=120
ok_marg = abs(da-db) > 0.30    # the two orderings must be clearly distinguishable
print(f"  Residential {100*chosen[0]:.2f}% > 35%  : {ok_res}")
print(f"  River       {100*chosen[2]:.2f}% < 5%   : {ok_riv}")
print(f"  margin decisive                 : {ok_marg}")
if ok_res and ok_riv and ok_marg:
    print("\nCHECK PASSED — palette ordering resolved. SWAP=True matches the documented")
    print("BGR convention, i.e. Residential is stored (128,0,0) and read as RGB (0,0,128).")
else:
    raise AssertionError(
        "Palette ordering could not be resolved. Do NOT proceed — a wrong ordering "
        "silently swaps Residential with River and Forest with UnusedLand, which are "
        "exactly the classes this notebook measures. Check that REAL_MSK points at the "
        "real mask folder and that the sample is not degenerate.")

def rgb_mask_to_index(path, size):
    m = np.array(Image.open(path).convert("RGB").resize((size,size), Image.NEAREST))
    return LUT[m[...,0], m[...,1], m[...,2]]

def index_mask_load(path, size):
    a = np.array(Image.open(path))
    if a.ndim == 3: return rgb_mask_to_index(path, size)
    a = np.array(Image.fromarray(a).resize((size,size), Image.NEAREST))
    a = a.copy(); a[a == 6] = 255          # S1 contract allowed a 7th index
    a[(a > 5) & (a != 255)] = 255
    return a

## 7. Block split and hash verification

Contiguous export-order blocks over the 408 base tiles, 2-tile buffer dropped from **TRAIN only**.
Reconstructed here; the count assertion `268 / 136` is the real gate. The sha1 is printed for
comparison against `eb0aebdd9919…` — if the serialisation differs the digest will differ even when
the split is identical, so **counts are authoritative, the hash is informational.**

In [ ]:
def block_split(n=cfg.N_BASE, folds=cfg.N_FOLDS, fold=cfg.FOLD, buf=cfg.BUFFER):
    ids = np.arange(1, n+1)
    edges = np.linspace(0, n, folds+1).astype(int)
    lo, hi = edges[fold], edges[fold+1]
    val   = ids[lo:hi]
    left  = ids[:lo][:-buf] if lo > 0 else ids[:0]
    right = ids[hi:][buf:]  if hi < n else ids[n:]
    train = np.concatenate([left, right])
    return train, val

TRAIN_IDS, VAL_IDS = block_split()
print(f"fold {cfg.FOLD}: train {len(TRAIN_IDS)} | val {len(VAL_IDS)}")
assert len(TRAIN_IDS) == cfg.EXPECT_TRAIN and len(VAL_IDS) == cfg.EXPECT_VAL, \
    "SPLIT MISMATCH — paste block_split() from your Stage-4 notebook over this cell"
h = hashlib.sha1(",".join(map(str, TRAIN_IDS.tolist())).encode()).hexdigest()
print("train sha1 (this serialisation):", h)
print("reference from the S4 notebooks :", cfg.TRAIN_SHA1_REF)
print("MATCH" if h == cfg.TRAIN_SHA1_REF else "digest differs — counts are correct, so this is a serialisation difference, not a split difference")
print("val ids:", VAL_IDS[:5], "...", VAL_IDS[-5:])

## 8. Datasets

Two sources, one interface. **Real val** = the 136 fold-1 originals (no flip variants — validation
never used them). **Fake** = the 804 `fake_pairs_R1` pairs, whose masks are already class-index PNGs
per the Stage-1/Stage-3 contract.

In [ ]:
from torch.utils.data import Dataset, DataLoader
IMNET_MEAN = np.array([0.485,0.456,0.406], np.float32)
IMNET_STD  = np.array([0.229,0.224,0.225], np.float32)

def load_img(path, size):
    im = Image.open(path).convert("RGB").resize((size,size), Image.BILINEAR)
    a = np.asarray(im, np.float32)/255.0
    a = (a - IMNET_MEAN)/IMNET_STD
    return torch.from_numpy(a.transpose(2,0,1))

class PairDS(Dataset):
    def __init__(self, pairs, size, mask_loader):
        self.pairs, self.size, self.ml = pairs, size, mask_loader
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        ip, mp = self.pairs[i]
        return load_img(ip, self.size), torch.from_numpy(self.ml(mp, self.size).astype(np.int64))

real_pairs = []
for i in VAL_IDS:
    ip, mp = REAL_IMG/f"image{i}.png", REAL_MSK/f"mask{i}.png"
    if ip.exists() and mp.exists(): real_pairs.append((ip, mp))
print("real val pairs:", len(real_pairs))

imgs = sorted([p for p in FAKE_DIR.rglob("*.png") if "mask" not in p.name.lower()])
def _twin(p):
    for cand in [p.parent/p.name.replace("image","mask"), p.parent/("mask_"+p.name),
                 p.parent.parent/"masks"/p.name, p.with_name(p.stem+"_mask.png")]:
        if cand.exists(): return cand
    return None
fake_pairs = [(p, _twin(p)) for p in imgs]
fake_pairs = [(a,b) for a,b in fake_pairs if b is not None]
print("fake pairs found:", len(fake_pairs), "(expected 804)")
assert len(fake_pairs) > 0, "no image/mask twins matched — inspect FAKE_DIR file naming and adjust _twin()"

DS_REAL = PairDS(real_pairs, cfg.IMG_SIZE, rgb_mask_to_index)
DS_FAKE = PairDS(fake_pairs, cfg.IMG_SIZE, index_mask_load)

## 9. Sanity checks on the synthetic shard

Class shares of the fake masks should sit close to the real fold-1 training marginal — that is what
Stage-3's greedy matching was for (KL 0.00003). If they do not, the wrong set is mounted.

In [ ]:
REAL_TRAIN_SHARE = np.array([52.53, 1.431, 0.469, 12.40, 14.27, 18.51])/100.0

def shares(ds, n=None):
    cnt = np.zeros(cfg.NUM_CLASSES, np.int64); n = n or len(ds)
    for i in range(min(n, len(ds))):
        m = ds[i][1].numpy()
        cnt += np.bincount(m[m != 255].ravel(), minlength=cfg.NUM_CLASSES)
    return cnt/max(cnt.sum(),1)

sf, sr = shares(DS_FAKE, 200), shares(DS_REAL, 136)
print(f"{'class':14s} {'fake%':>8s} {'realval%':>9s} {'realtrain%':>11s}")
for i,c in enumerate(CLASS_NAMES):
    print(f"{c:14s} {100*sf[i]:8.3f} {100*sr[i]:9.3f} {100*REAL_TRAIN_SHARE[i]:11.3f}")
kl = float(np.sum(np.where(sf>0, sf*np.log((sf+1e-12)/(REAL_TRAIN_SHARE+1e-12)), 0)))
print(f"\nKL(fake || real train) = {kl:.5f}   (Stage-3 reported 0.00003 for R1)")

# INDEPENDENT CROSS-CHECK on the palette decision made in section 6.
# The fake masks are class-index PNGs, so their class order involves no palette at
# all. Stage 3 matched their marginal to the real train marginal to KL 0.00003, so
# if section 6 had chosen the reversed ordering the two marginals would disagree
# violently on Residential and River. This confirms section 6 without reusing it.
rev = np.abs(sf - sr[[2,1,0,4,3,5]]).sum()
fwd = np.abs(sf - sr).sum()
print(f"\ncross-check vs the index-PNG fake masks:  L1 as-decoded {fwd:.4f}   reversed {rev:.4f}")
if fwd < rev:
    print("CONFIRMED — the real-mask decoding agrees with the palette-free fake masks.")
else:
    raise AssertionError("Palette decoding disagrees with the fake index masks — stop and re-check section 6.")
pres = np.mean([[ (DS_FAKE[i][1].numpy()==c).any() for c in range(6)] for i in range(min(200,len(DS_FAKE)))], 0)
print("presence rate  Road %.3f  River %.3f   (real 0.515 / 0.168)" % (pres[1], pres[2]))

## 10. Scorer registry and the generic checkpoint reader

Every Stage-4 `best.pt` holds `model_state` (EMA weights when EMA is on) plus `arch`, `arch_sig`,
`cfg` and `metrics`. The part-1 B0 checkpoint predates that convention, so the reader tries several
key names and strips the usual `module.` / `model.` prefixes.

In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files

def fetch(repo, patterns=("best.pt",)):
    try:
        files = list_repo_files(repo, token=HF_TOKEN)
    except Exception as e:
        print(f"  ! cannot list {repo}: {e}"); return None, []
    out = None
    for pat in patterns:
        for f in files:
            if f.endswith(pat):
                out = hf_hub_download(repo, f, token=HF_TOKEN); break
        if out: break
    return out, files

def read_state(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    if not isinstance(ck, dict): return ck, {}
    for k in ("model_state","state_dict","model","ema_state_dict","model_state_dict","weights"):
        if k in ck and isinstance(ck[k], dict):
            sd = ck[k]; break
    else:
        sd = ck
    clean = {}
    for k, v in sd.items():
        for pre in ("module.","model.","_orig_mod."):
            if k.startswith(pre): k = k[len(pre):]
        clean[k] = v
    meta = {k: ck[k] for k in ("arch","arch_sig","metrics") if k in ck}
    return clean, meta

SCORERS = {}   # name -> nn.Module (eval, on DEV)
BUILD_LOG = {}

## 11. Scorer A — SegFormer-B0, part-1 fold 1 (the control)

This is the instrument that produced the published Stage-3 numbers. Its only job here is to
reproduce them, which is what licenses trusting the other two.

In [ ]:
def build_b0():
    from transformers import SegformerForSemanticSegmentation
    ckpt, _ = fetch(cfg.REPO_B0, ("best.pt","pytorch_model.bin","model.safetensors"))
    if ckpt is None: raise RuntimeError("no checkpoint file found")
    m = SegformerForSemanticSegmentation.from_pretrained(
        "nvidia/segformer-b0-finetuned-ade-512-512",
        num_labels=cfg.NUM_CLASSES, ignore_mismatched_sizes=True)
    sd, meta = read_state(ckpt)
    missing, unexpected = m.load_state_dict(sd, strict=False)
    hit = 1 - len(missing)/max(len(m.state_dict()),1)
    print(f"  loaded {100*hit:.1f}% of tensors | missing {len(missing)} unexpected {len(unexpected)}")
    if hit < 0.90: raise RuntimeError(f"only {100*hit:.1f}% of tensors matched — wrong checkpoint layout")
    class W(nn.Module):
        def __init__(s, m): super().__init__(); s.m = m
        def forward(s, x):
            return F.interpolate(s.m(pixel_values=x).logits, size=x.shape[-2:],
                                 mode="bilinear", align_corners=False)
    return W(m)

if cfg.RUN_B0_CONTROL:
    try:
        SCORERS["SegFormer-B0 (part-1 f1, CONTROL)"] = build_b0().to(DEV).eval()
        BUILD_LOG["b0"] = "ok"; print("SegFormer-B0 control: BUILT")
    except Exception as e:
        BUILD_LOG["b0"] = str(e); print("SegFormer-B0 control FAILED:", e); traceback.print_exc(limit=2)

## 12. Scorer B — SegNeXt-T, Stage-4 arm R0 (the primary new instrument)

River IoU 0.4594 on real fold-1 — **3.2× the control**. The model class was pushed to the same repo
as `segnext_model.py`, so this one needs no pasted code.

Its LightHam decoder initialises NMF randomly at every forward pass, so its output is mildly
stochastic. `EVAL_REPEATS` below averages it, which also gives a noise floor for free.

In [ ]:
SEGNEXT_REPEATS = 3

def build_segnext():
    ckpt, files = fetch(cfg.REPO_SEGNEXT, ("best.pt",))
    if ckpt is None: raise RuntimeError("no best.pt")
    pys = [f for f in files if f.endswith(".py")]
    if not pys: raise RuntimeError("segnext_model.py not in repo — paste the class here instead")
    src = hf_hub_download(cfg.REPO_SEGNEXT, pys[0], token=HF_TOKEN)
    sys.path.insert(0, str(Path(src).parent))
    import importlib.util
    spec = importlib.util.spec_from_file_location("segnext_model", src)
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
    sd, meta = read_state(ckpt)
    build_fn = None
    for nm in ("build_model","SegNeXt","build_segnext","get_model","Model"):
        if hasattr(mod, nm): build_fn = getattr(mod, nm); break
    if build_fn is None: raise RuntimeError(f"no builder in {pys[0]}; found {dir(mod)[:20]}")
    try:    m = build_fn(num_classes=cfg.NUM_CLASSES)
    except TypeError:
        try:    m = build_fn(cfg.NUM_CLASSES)
        except TypeError: m = build_fn()
    missing, unexpected = m.load_state_dict(sd, strict=False)
    hit = 1 - len(missing)/max(len(m.state_dict()),1)
    print(f"  loaded {100*hit:.1f}% of tensors | arch_sig {meta.get('arch_sig')}")
    if hit < 0.95: raise RuntimeError(f"only {100*hit:.1f}% matched")
    class W(nn.Module):
        def __init__(s, m): super().__init__(); s.m = m
        def forward(s, x):
            y = s.m(x); y = y[0] if isinstance(y,(tuple,list)) else y
            if y.shape[-2:] != x.shape[-2:]:
                y = F.interpolate(y, size=x.shape[-2:], mode="bilinear", align_corners=False)
            return y
    return W(m)

if cfg.RUN_SEGNEXT:
    try:
        SCORERS["SegNeXt-T (S4 R0, PRIMARY)"] = build_segnext().to(DEV).eval()
        BUILD_LOG["segnext"] = "ok"; print("SegNeXt-T: BUILT")
    except Exception as e:
        BUILD_LOG["segnext"] = str(e); print("SegNeXt-T FAILED:", e); traceback.print_exc(limit=2)

## 13. Scorer C — TC-SegFormer, Stage-4 arm R0 (optional, needs a paste)

River IoU 0.4761, the best instrument available. It is **optional**: SegNeXt alone answers the
question, so if this cell is awkward, skip it and move on.

The notebook first checks whether the repo ships a `.py`. If it does not, paste the `TCSegFormer`
class definition from **section 6 of `Stage4-tcsegformer-R0R1.ipynb`** into the marked block below —
only the model class and whatever it imports, nothing else. Note the naming trap already recorded in
the project: part-1's standalone loader called the H/2 final conv `classifier` while training calls
it `refine`, so `strict=True` fails on every real checkpoint. This loader uses `strict=False` and
reports the hit rate instead.

In [ ]:
# ============================ PASTE-IN BLOCK ==================================
# Leave empty to auto-detect a .py in the repo. If detection fails, paste the
# TCSegFormer class (and its helpers) from section 6 of your S4 TC notebook here.
TC_MODEL_SRC = r'''
'''
# ==============================================================================

def build_tc():
    ckpt, files = fetch(cfg.REPO_TC, ("best.pt",))
    if ckpt is None: raise RuntimeError("no best.pt")
    ns = {}
    pys = [f for f in files if f.endswith(".py")]
    if TC_MODEL_SRC.strip():
        exec(compile(TC_MODEL_SRC, "<tc_paste>", "exec"), ns)
        print("  using pasted source")
    elif pys:
        src = hf_hub_download(cfg.REPO_TC, pys[0], token=HF_TOKEN)
        exec(compile(open(src).read(), pys[0], "exec"), ns)
        print(f"  using {pys[0]} from the repo")
    else:
        raise RuntimeError("no .py in repo and TC_MODEL_SRC is empty — skipping (this is fine)")
    build_fn = None
    for nm in ("build_model","TCSegFormer","TCSegformer","build_tc","get_model","Model"):
        if nm in ns: build_fn = ns[nm]; break
    if build_fn is None: raise RuntimeError("no TCSegFormer/build_model in the pasted source")
    sd, meta = read_state(ckpt)
    for kwargs in ({"num_classes": cfg.NUM_CLASSES}, {"n_classes": cfg.NUM_CLASSES}, {}):
        try: m = build_fn(**kwargs); break
        except TypeError: continue
    else: m = build_fn(cfg.NUM_CLASSES)
    missing, unexpected = m.load_state_dict(sd, strict=False)
    hit = 1 - len(missing)/max(len(m.state_dict()),1)
    print(f"  loaded {100*hit:.1f}% of tensors | missing[:4] {list(missing)[:4]}")
    if hit < 0.90: raise RuntimeError(f"only {100*hit:.1f}% matched")
    class W(nn.Module):
        def __init__(s, m): super().__init__(); s.m = m
        def forward(s, x):
            y = s.m(x); y = y[0] if isinstance(y,(tuple,list)) else y
            if hasattr(y, "logits"): y = y.logits
            if y.shape[-2:] != x.shape[-2:]:
                y = F.interpolate(y, size=x.shape[-2:], mode="bilinear", align_corners=False)
            return y
    return W(m)

if cfg.RUN_TC:
    try:
        SCORERS["TC-SegFormer (S4 R0)"] = build_tc().to(DEV).eval()
        BUILD_LOG["tc"] = "ok"; print("TC-SegFormer: BUILT")
    except Exception as e:
        BUILD_LOG["tc"] = str(e); print("TC-SegFormer skipped:", e)

## 14. Build report — the notebook degrades, it does not fail

Design rule carried over from the Stage-3 multi-GPU episode: **an optional component must never be
able to take down the thing it was meant to improve.** If only SegNeXt built, the question is still
answered.

In [ ]:
print("scorers built:", len(SCORERS))
for k in SCORERS: print("  +", k)
for k, v in BUILD_LOG.items():
    if v != "ok": print(f"  - {k}: {v[:140]}")
assert len(SCORERS) > 0, "no scorer built — nothing can be measured; check the repo ids in CFG"
if "SegNeXt-T (S4 R0, PRIMARY)" not in SCORERS:
    print("\nWARNING: the primary instrument did not build. Results will be partial.")

## 15. Smoke test — 4 real tiles through every scorer

Catches shape, normalisation and class-order faults in seconds instead of after the full pass.

In [ ]:
xb = torch.stack([DS_REAL[i][0] for i in range(4)]).to(DEV)
yb = torch.stack([DS_REAL[i][1] for i in range(4)])
for name, m in SCORERS.items():
    with torch.no_grad(), torch.autocast("cuda", enabled=(cfg.AMP and DEV.type=="cuda")):
        out = m(xb)
    pr = out.argmax(1).cpu()
    acc = float((pr[yb!=255] == yb[yb!=255]).float().mean())
    print(f"{name:38s} logits {tuple(out.shape)}  pixel-acc {acc:.4f}  pred classes {sorted(set(pr.unique().tolist()))}")
print("\npixel-acc well above 0.5 means the class order matches; near 0.17 would mean a permutation")

## 16. Metrics — pooled confusion matrix, IoU, F1, clDice

Pooled (dataset-level), not tile-averaged, which is the convention every GeoNUSAF number uses —
and the reason per-class F1 satisfies `F1 = 2·IoU/(1+IoU)` exactly in the model cards. clDice is
computed per image over Road and River and averaged across images where the class is present in the
ground truth, which is the Stage-3 convention.

In [ ]:
from skimage.morphology import skeletonize

def cm_update(cm, pred, gt):
    v = gt != cfg.IGNORE_INDEX
    idx = gt[v].astype(np.int64)*cfg.NUM_CLASSES + pred[v].astype(np.int64)
    cm += np.bincount(idx, minlength=cfg.NUM_CLASSES**2).reshape(cfg.NUM_CLASSES, cfg.NUM_CLASSES)
    return cm

def cm_metrics(cm):
    tp = np.diag(cm).astype(np.float64)
    fp = cm.sum(0)-tp; fn = cm.sum(1)-tp
    iou = tp/np.maximum(tp+fp+fn, 1e-9)
    f1  = 2*tp/np.maximum(2*tp+fp+fn, 1e-9)
    ua  = tp/np.maximum(tp+fp, 1e-9)     # precision
    pa  = tp/np.maximum(tp+fn, 1e-9)     # recall
    oa  = tp.sum()/max(cm.sum(), 1e-9)
    pe  = float((cm.sum(0)/max(cm.sum(),1e-9) * cm.sum(1)/max(cm.sum(),1e-9)).sum())
    kap = (oa-pe)/max(1-pe, 1e-9)
    return dict(iou=iou, f1=f1, ua=ua, pa=pa, mIoU=float(iou.mean()),
                mF1=float(f1.mean()), OA=float(oa), kappa=float(kap), pe=pe)

def cldice_one(pred, gt, c):
    P, G = (pred==c), (gt==c)
    if G.sum()==0: return None
    if P.sum()==0: return 0.0
    sp, sg = skeletonize(P), skeletonize(G)
    tprec = (sp & G).sum()/max(sp.sum(),1)
    tsens = (sg & P).sum()/max(sg.sum(),1)
    return float(0.0 if (tprec+tsens)==0 else 2*tprec*tsens/(tprec+tsens))

## 17. Evaluation loop

One pass per (scorer, source). `repeats > 1` averages the confusion matrices for SegNeXt's stochastic
NMF and records the per-repeat mIoU spread — that spread **is** the noise floor the master document
asks for next to the SegNeXt delta.

In [ ]:
@torch.no_grad()
def evaluate(model, ds, repeats=1, tag=""):
    dl = DataLoader(ds, batch_size=cfg.BATCH, shuffle=False,
                    num_workers=cfg.NUM_WORKERS, pin_memory=(DEV.type=="cuda"))
    cms, per_rep = [], []
    cld = {c: [] for c in cfg.CLDICE_CLASSES}
    for r in range(repeats):
        cm = np.zeros((cfg.NUM_CLASSES, cfg.NUM_CLASSES), np.int64)
        t0 = time.time()
        for bi,(x,y) in enumerate(dl):
            x = x.to(DEV, non_blocking=True)
            with torch.autocast("cuda", enabled=(cfg.AMP and DEV.type=="cuda")):
                pr = model(x).argmax(1)
            pr = pr.cpu().numpy(); gt = y.numpy()
            for k in range(pr.shape[0]):
                cm = cm_update(cm, pr[k], gt[k])
                if r == 0:
                    for c in cfg.CLDICE_CLASSES:
                        v = cldice_one(pr[k], gt[k], c)
                        if v is not None: cld[c].append(v)
            if bi % 25 == 0:
                print(f"    {tag} rep{r} batch {bi}/{len(dl)}  {time.time()-t0:.0f}s", end="\r")
        cms.append(cm); per_rep.append(cm_metrics(cm)["mIoU"])
    cm = np.sum(cms, 0)
    out = cm_metrics(cm)
    out["cm"] = cm.tolist()
    out["mIoU_repeats"] = per_rep
    out["mIoU_std"] = float(np.std(per_rep)) if len(per_rep) > 1 else None
    out["cldice"] = {CLASS_NAMES[c]: (float(np.mean(cld[c])) if cld[c] else None) for c in cfg.CLDICE_CLASSES}
    print(f"    {tag}: mIoU {out['mIoU']:.4f}  OA {out['OA']:.4f}" + (
          f"  (mIoU sd over {repeats} repeats {out['mIoU_std']:.4f})" if out["mIoU_std"] else ""))
    return out

## 18. Run — real fold-1 validation imagery (the per-scorer ceiling)

In [ ]:
RESULTS = {}
for name, m in SCORERS.items():
    reps = SEGNEXT_REPEATS if name.startswith("SegNeXt") else 1
    print(f"\n[real val] {name}")
    RESULTS[(name,"real_val")] = evaluate(m, DS_REAL, repeats=reps, tag="real")

## 19. Run — the synthetic pool `fake_pairs_R1`

The 804 pairs Stage 4 actually trained on, not the full 4824-pair pool.

In [ ]:
for name, m in SCORERS.items():
    reps = SEGNEXT_REPEATS if name.startswith("SegNeXt") else 1
    print(f"\n[fake R1] {name}")
    RESULTS[(name,"fake_R1")] = evaluate(m, DS_FAKE, repeats=reps, tag="fake")

## 20. Layout fidelity, ceiling and ratio — per class, per scorer

`ratio = fake / real`. A ratio near 1.0 means the generator reproduces that class as well as the
scorer can read it on real imagery; a ratio near 0 means the class is not being generated at all.

In [ ]:
def show(name):
    fk, rl = RESULTS[(name,"fake_R1")], RESULTS[(name,"real_val")]
    print("="*92); print(name); print("="*92)
    print(f"{'class':14s} {'fake IoU':>10s} {'real IoU':>10s} {'ratio':>8s} {'fake F1':>9s} {'real F1':>9s}")
    for i,c in enumerate(CLASS_NAMES):
        r = fk['iou'][i]/max(rl['iou'][i],1e-9)
        print(f"{c:14s} {fk['iou'][i]:10.4f} {rl['iou'][i]:10.4f} {r:8.3f} {fk['f1'][i]:9.4f} {rl['f1'][i]:9.4f}")
    print(f"{'mIoU':14s} {fk['mIoU']:10.4f} {rl['mIoU']:10.4f} {fk['mIoU']/max(rl['mIoU'],1e-9):8.3f}")
    for c in ("Road","River"):
        a,b = fk['cldice'].get(c), rl['cldice'].get(c)
        if a is not None and b is not None:
            print(f"{'clDice '+c:14s} {a:10.4f} {b:10.4f} {a/max(b,1e-9):8.3f}")
    print()

for name in SCORERS: show(name)

## 21. THE GATE — does the control reproduce the published Stage-3 numbers?

Stage 3 reported **fake mIoU 0.4395** and **real ceiling 0.4646** with this scorer, giving ratio
0.946, and **River 0.033–0.055** on fakes against **0.1434** on real.

If the control lands within tolerance, the harness is faithful and everything in §20 is
trustworthy. **If it does not, stop — the new numbers mean nothing and the discrepancy must be
chased first.**

In [ ]:
GATE = None
ctrl = [n for n in SCORERS if "CONTROL" in n]
if not ctrl:
    print("control scorer did not build — the new numbers cannot be validated against Stage 3.")
    print("They are still informative, but say so explicitly in the paper.")
else:
    n = ctrl[0]
    fk, rl = RESULTS[(n,"fake_R1")], RESULTS[(n,"real_val")]
    df, dr = abs(fk['mIoU']-cfg.REF_FAKE_MIOU), abs(rl['mIoU']-cfg.REF_REAL_MIOU)
    print(f"fake mIoU  measured {fk['mIoU']:.4f}  published {cfg.REF_FAKE_MIOU:.4f}  |diff| {df:.4f}")
    print(f"real mIoU  measured {rl['mIoU']:.4f}  published {cfg.REF_REAL_MIOU:.4f}  |diff| {dr:.4f}")
    print(f"River      measured {fk['iou'][2]:.4f} on fakes vs published 0.033-0.055")
    print(f"           measured {rl['iou'][2]:.4f} on real  vs published 0.1434")
    GATE = (df <= cfg.REF_TOL) and (dr <= cfg.REF_TOL)
    print("\nGATE:", "PASSED — harness reproduces Stage 3" if GATE else
          "FAILED — do not use the new numbers until this is explained")
    if not GATE:
        print("  likely causes, in order: wrong fake set mounted (R2/R3 instead of R1);")
        print("  palette LUT swapped (check the SWAP print in section 6);")
        print("  a different checkpoint in the B0 repo than the one Stage 3 used.")

## 22. Recomputed contamination estimates for the master document

This is the block that goes straight into Table 11.5d. Contamination is
`0.5 × (1 − layout IoU)` — at a 50/50 row mixture, roughly half of a class's supervision comes from
generated imagery, of which a fraction `(1 − layout IoU)` does not read as that class.

The mean Stage-4 per-class deltas are fixed constants from the R1 arm; only the fidelity and
contamination columns can change.

In [ ]:
MEAN_DELTA = {"Residential":-0.0093,"Road":-0.0303,"River":-0.0566,
              "Forest":-0.0048,"UnusedLand":+0.0137,"Agricultural":+0.0062}
OLD_FID    = {"Residential":0.820,"Road":0.230,"River":0.044,
              "Forest":0.665,"UnusedLand":0.320,"Agricultural":0.535}

prim = ([n for n in SCORERS if "PRIMARY" in n] or [n for n in SCORERS if "TC-" in n] or list(SCORERS))[0]
fk = RESULTS[(prim,"fake_R1")]
print(f"instrument: {prim}\n")
print(f"{'class':14s} {'oldFid':>8s} {'newFid':>8s} {'oldCont':>9s} {'newCont':>9s} {'meanDelta':>10s}")
rows = []
for i,c in enumerate(CLASS_NAMES):
    nf = float(fk['iou'][i]); oc = 0.5*(1-OLD_FID[c]); nc = 0.5*(1-nf)
    print(f"{c:14s} {OLD_FID[c]:8.3f} {nf:8.3f} {100*oc:8.1f}% {100*nc:8.1f}% {MEAN_DELTA[c]:+10.4f}")
    rows.append(dict(cls=c, old_fidelity=OLD_FID[c], new_fidelity=nf,
                     old_contam=oc, new_contam=nc, mean_delta=MEAN_DELTA[c]))
x = np.array([r["new_contam"] for r in rows]); y = np.array([r["mean_delta"] for r in rows])
print(f"\nPearson r (contamination vs mean delta), all six classes: {np.corrcoef(x,y)[0,1]:+.3f}")
k = [i for i,r in enumerate(rows) if r["cls"] in ("Residential","Road","River")]
print(f"Pearson r over the appearance/topology classes only:      {np.corrcoef(x[k],y[k])[0,1]:+.3f}")
print("\nHow to read it: a strongly negative r over Residential/Road/River supports the")
print("proportionality claim in §22.7. UnusedLand and Agricultural are expected to break it —")
print("that is the 'layout transfers, appearance does not' argument, not a failure.")

## 23. FREE RETRIEVAL — pull `results_*.json` from all ten Stage-4 repos

No GPU, no inference. These files carry the four metric families missing from every model card:
**FWIoU, clDice for Road and River, efficiency, and confusion matrices** — plus, if present,
`stage4_R0_vs_R1.json` (the paired-bootstrap intervals) and SegNeXt's `nmf_eval_mIoU_std`.

The clDice columns matter most: if the four non-TC architectures lose more clDice than IoU under R1,
that is independent support for the soft-clDice mechanism in §22.10, from a different metric family.

In [ ]:
HARVEST = {}
for repo in cfg.S4_REPOS:
    try:
        files = list_repo_files(repo, token=HF_TOKEN)
    except Exception as e:
        print(f"{repo:58s} UNREACHABLE ({str(e)[:40]})"); continue
    want = [f for f in files if f.endswith(".json") or f.endswith(".csv")]
    got = {}
    for f in want:
        try:
            p = hf_hub_download(repo, f, token=HF_TOKEN)
            if f.endswith(".json"):
                got[f] = json.load(open(p))
            else:
                shutil.copy(p, Path(cfg.OUT_DIR)/f"{repo.split('/')[-1]}__{Path(f).name}")
                got[f] = "csv saved"
        except Exception as e:
            got[f] = f"ERR {str(e)[:40]}"
    HARVEST[repo] = got
    flags = []
    if any("R0_vs_R1" in f for f in got): flags.append("BOOTSTRAP")
    if any("cldice" in json.dumps(v).lower() for v in got.values() if isinstance(v,(dict,list))): flags.append("clDice")
    if any("nmf" in json.dumps(v).lower() for v in got.values() if isinstance(v,(dict,list))): flags.append("nmf_std")
    print(f"{repo:58s} {len(got):2d} files  {' '.join(flags)}")

json.dump(HARVEST, open(Path(cfg.OUT_DIR)/"stage4_harvest.json","w"), indent=2, default=str)
print("\nsaved stage4_harvest.json")

## 24. Save everything

In [ ]:
payload = dict(
    generated_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    cfg={k:(list(v) if isinstance(v,tuple) else v) for k,v in asdict(cfg).items()},
    split=dict(n_train=int(len(TRAIN_IDS)), n_val=int(len(VAL_IDS)), train_sha1=h,
               train_sha1_ref=cfg.TRAIN_SHA1_REF),
    n_fake=len(fake_pairs), n_real=len(real_pairs), lut_swapped=bool(SWAP),
    build_log=BUILD_LOG, gate_passed=GATE, class_names=CLASS_NAMES,
    results={f"{a}||{b}": {k:(v.tolist() if isinstance(v,np.ndarray) else v)
                           for k,v in r.items()} for (a,b), r in RESULTS.items()},
)
jp = Path(cfg.OUT_DIR)/"rescore_layout_fidelity.json"
json.dump(payload, open(jp,"w"), indent=2, default=str)

import csv
cp = Path(cfg.OUT_DIR)/"rescore_summary.csv"
with open(cp,"w",newline="") as f:
    w = csv.writer(f); w.writerow(["scorer","source","class","IoU","F1","UA","PA","clDice"])
    for (nm,src), r in RESULTS.items():
        for i,c in enumerate(CLASS_NAMES):
            w.writerow([nm,src,c,f"{r['iou'][i]:.6f}",f"{r['f1'][i]:.6f}",
                        f"{r['ua'][i]:.6f}",f"{r['pa'][i]:.6f}",
                        r["cldice"].get(c,"") if r["cldice"].get(c) is not None else ""])
        w.writerow([nm,src,"__MEAN__",f"{r['mIoU']:.6f}",f"{r['mF1']:.6f}","","",""])
print("saved:"); [print("  ", p) for p in sorted(Path(cfg.OUT_DIR).iterdir())]

## 25. Checklist — what to send back

Send these three things and the master document can be updated without any further round trip:

1. **`rescore_layout_fidelity.json`** — everything, including per-scorer confusion matrices.
2. **`rescore_summary.csv`** — the flat per-class table.
3. **The §21 gate verdict and the §20 tables**, pasted as text.

Also worth mentioning in your message:

- whether the **§21 gate passed** (if it failed, nothing downstream is usable);
- which scorers built and which did not (§14);
- whether **§23 found any `stage4_R0_vs_R1.json`** — those are the bootstrap intervals that unblock
  every `PENDING` cell in Table 11.4, and whether it found `nmf_eval_mIoU_std`;
- whether **§23 found clDice** for Road and River in both arms.

**What will change in the paper depending on the outcome.** If River fidelity stays near zero under
a River-capable scorer, §22.7's contamination argument holds and is now measured with an instrument
that could have detected the signal. If it rises substantially, the River contamination estimate
falls, the strict proportionality claim in §22.7 weakens, and the argument shifts weight onto the
presence-prior mechanism in §22.8 — which is separately measured and does not depend on this number
at all. Either way the paper gets more defensible; there is no outcome here that hurts.

In [ ]:
print("RESCORE COMPLETE")
print(f"  scorers built : {len(SCORERS)}  -> {list(SCORERS)}")
print(f"  gate passed   : {GATE}")
print(f"  outputs       : {cfg.OUT_DIR}")
print("\nSend rescore_layout_fidelity.json + rescore_summary.csv + the section 20/21 printouts.")